Extract adatas from .zarr files

In [1]:
import spatialdata as sd
#import spatialdata_plot 
import os
import scanpy as sc
import numpy as np
from pathlib import Path
import rapids_singlecell as rsc
import pandas as pd
from scipy.sparse import issparse

/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MAIN_DIR = '/mnt/data/project0062/proseg_data'
os.chdir(MAIN_DIR)
os.getcwd()

'/mnt/autofs/data/userdata/project0062/proseg_data'

In [3]:
slide01_path = "data/slides/SLIDE01/zarr/SLIDE01_proseg.zarr"
slide02_path = "data/slides/SLIDE02/zarr/SLIDE02_proseg.zarr"
slide03_path = "data/slides/SLIDE03/zarr/SLIDE03_proseg.zarr"
slide04_path = "data/slides/SLIDE04/zarr/SLIDE04_proseg.zarr"

In [4]:
sdata01 = sd.read_zarr(slide01_path)
sdata02 = sd.read_zarr(slide02_path)
sdata03 = sd.read_zarr(slide03_path)
sdata04 = sd.read_zarr(slide04_path)

/tmp/ipykernel_2854254/4244499838.py:1: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata01 = sd.read_zarr(slide01_path)
/tmp/ipykernel_2854254/4244499838.py:2: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata02 = sd.read_zarr(slide02_path)
/tmp/ipykernel_2854254/4244499838.py:3: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata03 = sd.read_zarr(slide03_path)
/tmp/ipykernel_2854254/4244499838.py:4: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata04 = sd.read_zarr(slide04_path)


### Extract polygons

In [5]:
print(list(sdata01.shapes.keys()))

['cell_boundaries']


In [6]:
polygons_sdata01 = sdata01.shapes["cell_boundaries"].copy()
polygons_sdata02 = sdata02.shapes["cell_boundaries"].copy()
polygons_sdata03 = sdata03.shapes["cell_boundaries"].copy()
polygons_sdata04 = sdata04.shapes["cell_boundaries"].copy()

In [7]:
polygons_sdata01["cell_id"] = ("SLIDE01_" + polygons_sdata01["cell"].astype(str))
polygons_sdata02["cell_id"] = ("SLIDE02_" + polygons_sdata01["cell"].astype(str))
polygons_sdata03["cell_id"] = ("SLIDE03_" + polygons_sdata01["cell"].astype(str))
polygons_sdata04["cell_id"] = ("SLIDE04_" + polygons_sdata01["cell"].astype(str))

In [8]:
polygons_sdata01.head()

,cell,geometry,cell_id
0,0,"MULTIPOLYGON (((610 1387.08838, 610 1388.08838...",SLIDE01_0
1,1,"MULTIPOLYGON (((632 1389.08838, 632 1391.08838...",SLIDE01_1
2,2,"MULTIPOLYGON (((512 1468.08838, 512 1484.08838...",SLIDE01_2
3,3,"MULTIPOLYGON (((518 1470.08838, 518 1475.08838...",SLIDE01_3
4,4,"MULTIPOLYGON (((518 1483.08838, 518 1482.08838...",SLIDE01_4


In [9]:
# combine all 
polygons_comb = pd.concat(
    [
        polygons_sdata01,
        polygons_sdata02,
        polygons_sdata03,
        polygons_sdata04,
    ],
    ignore_index=True,
)

polygons_comb.head()

,cell,geometry,cell_id
0,0,"MULTIPOLYGON (((610 1387.08838, 610 1388.08838...",SLIDE01_0
1,1,"MULTIPOLYGON (((632 1389.08838, 632 1391.08838...",SLIDE01_1
2,2,"MULTIPOLYGON (((512 1468.08838, 512 1484.08838...",SLIDE01_2
3,3,"MULTIPOLYGON (((518 1470.08838, 518 1475.08838...",SLIDE01_3
4,4,"MULTIPOLYGON (((518 1483.08838, 518 1482.08838...",SLIDE01_4


In [11]:
polygons_comb_path = Path("/mnt/data/project0062/proseg_data/data/comb/polygons/comb-polygons-light.parquet")
# create parent path 
polygons_comb_path.parent.mkdir(parents=True, exist_ok=True)

# save
polygons_comb.to_parquet(
    polygons_comb_path,
    index=False,
)

### code to test polygons

In [ ]:
# import plotnine as p9
# fig_size=(20,20)
# background_color='black'
# p = (
#         p9.ggplot(polygons_sdata01)
#         + p9.geom_map(
#             p9.aes(geometry="geometry"),
#             fill=None,
#             color="white",
#             size=0.1,
#         )
#         + p9.coord_fixed(1)  # keep aspect ratio
#         #+ p9.scale_fill_manual(values=colors)
#         + p9.theme(
#             axis_line=p9.element_blank(),
#             axis_text=p9.element_blank(),
#             axis_ticks=p9.element_blank(),
#             axis_title=p9.element_blank(),
#             panel_background=p9.element_rect(fill=background_color),
#             panel_grid_major=p9.element_blank(),
#             panel_grid_minor=p9.element_blank(), 
#             figure_size=fig_size,
#         )
#     )

# p

In [ ]:
import geopandas as gpd